In [ ]:
import pandas as pd
import numpy as np
from IPython.core.pylabtools import figsize
from PIL.ImageChops import difference
from matplotlib import pyplot as plt
from scipy.signal.windows import blackman
from scipy.spatial import distance
from statsmodels.graphics.tukeyplot import results
import os
import re
from utils import *

from statsmodels.sandbox.distributions.genpareto import shape

In [ ]:
# df_hme1 = pd.read_excel("Experiment/hme1_1/Data/Processed/Full_dataset_1_hTERT_HME1_diff.xlsx")
df_hme2 = pd.read_excel("Experiment/hme1_1/Data/Processed/Full_dataset_1_hTERT_HME1_functional_names_diff_scaled.xlsx")
# df_hek = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")

# df_hme1 = df_hme1.fillna(0)
df_hme2 = df_hme2.fillna(0)
# df_hek = df_hek.fillna(0)

In [ ]:
plot_protein_profile(df=df_hme2, experiment="hme1_2", data_type= "log2_FC",
                     proteins = ['GAB1', "EGFR", 'MAP2K1'],
                     saving_path= "", saving_info="",
                     legend=False,
                     save_pdf=False, save_png=False)

In [ ]:
plot_protein_profiles_fine_line(df = df_hme2, proteins = ['GAB1', "EGFR"], data_type= "log2_FC",
                         saving_path = str, legend = False,
                         save_pdf=False, save_png=False)

In [ ]:
# #### Calculating means
# dictionary = {}
# for index, row in df.iterrows():
#     EGF = row[["FC_EGF_full", "FC_EGF_starve", "FC_EGF1", "FC_EGF2", "FC_EGF5", "FC_EGF10",  "FC_EGF90"]]
#     INS = row[["FC_INS_full", "FC_INS_starve", "FC_INS1", "FC_INS2", "FC_INS5", "FC_INS10", "FC_INS90"]]
#     EGFnINS = row[["FC_EGFnINS_full", "FC_EGFnINS_starve", "FC_EGFnINS1", "FC_EGFnINS2", "FC_EGFnINS5", "FC_EGFnINS10", "FC_EGFnINS90"]]
#     results = []
#     for i in range(len(EGF)):
#         results.append(INS[i]-EGFnINS[i])
#     sumatory = sum([abs(x) for x in results])
#     dictionary[(row["site"])] = sumatory
#
# keys = list(dictionary.keys())
# values = list(dictionary.values())
# sorted_value_index = np.argsort(values)
# sorted_dict = {keys[i]: values[i] for i in sorted_value_index}
#
# sorted_dict
# keys_sorted = list(sorted_dict.keys())
# values_sorted = list(sorted_dict.values())
#
# diff_df = pd.DataFrame(columns=["site", "difference_INS_EGFnINS" ]) #"protein_Id",
#
# for key in keys_sorted:
#     diff_df.loc[len(diff_df)] = [(key.split("_"))[0],key, sorted_dict[key]]
# diff_df

# #### Calaculating geometric mean for differences
# dictionary = {}
# for index, row in df.iterrows():
#     geometric_mean = np.sqrt(row["difference_EGF_vs_EGFnINS"]*row["difference_INS_vs_EGFnINS"])
#     dictionary[(row["site"])] = geometric_mean
#
# geometric_mean_df = pd.DataFrame(columns=["site", "difference_geometric_mean"])
# for key in dictionary:
#     geometric_mean_df.loc[len(geometric_mean_df)] = [key, dictionary[key]]
# geometric_mean_df

In [ ]:
# #### Calculating means
# data_with_df = pd.merge(df, diff_df, on="site", how="left")
# data_with_df.to_excel("Experiment/hme1_1/Data/Processed/Full_dataset_1_hTERT_HME1_diff.xlsx", index = False)
# data_with_df

# #### Calaculating geometric mean for differences
data_with_geometric_mean = pd.merge(df, geometric_mean_df, on="site", how="left")
# data_with_geometric_mean.to_excel("Experiment/hek_1/Data/Processed/Full_dataset_HEK293T_diff.xlsx", index=False)


In [ ]:
# data_with_df = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")
data_with_df = data_with_geometric_mean.copy()


In [ ]:

data_with_df_EGF = data_with_df.sort_values(by=["difference_EGF_vs_EGFnINS"], ascending=False)
data_with_df_INS = data_with_df.sort_values(by=["difference_INS_vs_EGFnINS"], ascending=False)
data_with_df_geometric_mean = data_with_df.sort_values(by=["difference_geometric_mean"], ascending=False)
# sites_values = data_with_df.site.values
difference_values_EGF = data_with_df_EGF.difference_EGF_vs_EGFnINS.values
difference_values_INS = data_with_df_INS.difference_INS_vs_EGFnINS.values
difference_values_geometric_mean = data_with_df_geometric_mean.difference_geometric_mean.values

In [ ]:
fig, ax = plt.subplots(3, 1, figsize = (12,12))
    
ax[0].bar(range(len(difference_values_EGF)), difference_values_EGF)
ax[0].set_ylabel("Difference between log2FC EGF and EGFnINS")
# ax[0].axhline(value_threshold, color='red', linestyle='--', linewidth=0.5)

ax[1].bar(range(len(difference_values_INS)), difference_values_INS)
ax[1].set_ylabel("Difference between log2FC INS and EGFnINS")
# ax[1].axhline(value_threshold, color='red', linestyle='--', linewidth=0.5)


ax[2].bar(range(len(difference_values_geometric_mean)), difference_values_geometric_mean)
ax[2].set_ylabel("Geometric mean of differences")
# ax[2].axhline(value_threshold, color='red', linestyle='--', linewidth=0.5)

fig.suptitle("hek_1 (all dataset, no filtering)", weight='bold')
fig.tight_layout()
# plt.savefig(f".pdf")

In [ ]:
subset_df_EGF = data_with_df_EGF.loc[data_with_df_EGF["n_rep"] > 3]
subset_df_INS = data_with_df_INS.loc[data_with_df_INS["n_rep"] > 3]
subset_df_geometric_mean = data_with_df_geometric_mean.loc[data_with_df_geometric_mean["n_rep"] > 3]

print(subset_df_EGF.shape)
subset_df_EGF = subset_df_EGF.sort_values(by=['difference_EGF_vs_EGFnINS'], ascending=False)
subset_df_INS = subset_df_INS.sort_values(by=['difference_INS_vs_EGFnINS'], ascending=False)
subset_df_geometric_mean = subset_df_geometric_mean.sort_values(by=['difference_geometric_mean'], ascending=False)

df_EGF = subset_df_EGF.difference_EGF_vs_EGFnINS.values
df_INS = subset_df_INS.difference_INS_vs_EGFnINS.values
df_geometric_mean = subset_df_geometric_mean.difference_geometric_mean.values

In [ ]:

fig, ax = plt.subplots(3, 1, figsize = (12,12))
    
ax[0].bar(range(len(df_EGF)), df_EGF)
ax[0].set_ylabel("Difference between log2FC EGF and EGFnINS")

ax[1].bar(range(len(df_INS)), df_INS)
ax[1].set_ylabel("Difference between log2FC INS and EGFnINS")

ax[2].bar(range(len(df_geometric_mean)), df_geometric_mean)
ax[2].set_ylabel("Geometric mean of differences")

fig.suptitle("hek_1 sites with more than 1 replicate", weight='bold')
fig.tight_layout()

# plt.show()

# Check euclidean distances


In [ ]:
df = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")
df = df.fillna(0)
df

In [ ]:

#### Calculating euclidean distance
dictionary = {}
for index, row in df.iterrows():
    EGF = row[["FC_EGF_full", "FC_EGF_starve", "FC_EGF1", "FC_EGF2", "FC_EGF5", "FC_EGF10",  "FC_EGF90"]]
    INS = row[["FC_INS_full", "FC_INS_starve", "FC_INS1", "FC_INS2", "FC_INS5", "FC_INS10", "FC_INS90"]]
    EGFnINS = row[["FC_EGFnINS_full", "FC_EGFnINS_starve", "FC_EGFnINS1", "FC_EGFnINS2", "FC_EGFnINS5", "FC_EGFnINS10", "FC_EGFnINS90"]]

    EGFvsINS
    EGF
    for i in range(len(EGF)):
        results.append(INS[i]-EGFnINS[i])
    sumatory = sum([abs(x) for x in results])
    dictionary[(row["site"])] = sumatory

keys = list(dictionary.keys())
values = list(dictionary.values())
sorted_value_index = np.argsort(values)
sorted_dict = {keys[i]: values[i] for i in sorted_value_index}

sorted_dict
keys_sorted = list(sorted_dict.keys())
values_sorted = list(sorted_dict.values())

diff_df = pd.DataFrame(columns=["site", "difference_INS_EGFnINS" ]) #"protein_Id",

for key in keys_sorted:
    diff_df.loc[len(diff_df)] = [(key.split("_"))[0],key, sorted_dict[key]]
diff_df

# # #### Calaculating geometric mean for differences
# dictionary = {}
# for index, row in df.iterrows():
#     geometric_mean = np.sqrt(row["difference_EGF_vs_EGFnINS"]*row["difference_INS_vs_EGFnINS"])
#     dictionary[(row["site"])] = geometric_mean
#
# geometric_mean_df = pd.DataFrame(columns=["site", "difference_geometric_mean"])
# for key in dictionary:
#     geometric_mean_df.loc[len(geometric_mean_df)] = [key, dictionary[key]]
# geometric_mean_df

In [ ]:
df = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")
df

In [ ]:
#Checking the max distance of the minimun distance
dictionary ={}

for index, row in df.iterrows():
    if row["difference_INS_vs_EGFnINS"] > row["difference_EGF_vs_EGFnINS"]:
        dictionary[(row["site"])] = row["difference_EGF_vs_EGFnINS"]
    else:
        dictionary[(row["site"])] = row["difference_INS_vs_EGFnINS"]

min_diff_df = pd.DataFrame(columns=["site", "min_diff"])

for key in dictionary:
    min_diff_df.loc[len(min_diff_df)] = [key, dictionary[key]]

# #### merging df
data_with_min_diff = pd.merge(df, min_diff_df, on="site", how="left")
data_with_min_diff

In [ ]:
# data_with_min_diff.to_excel("Experiment/hek_1/Data/Processed/Full_dataset_HEK293T_diff.xlsx", index = False)

In [ ]:
# Calculating the total protein score of change under co-stimulation
prot_list = list(df.protein_Id.unique())

protein_score_df = pd.DataFrame(columns=["protein_Id", "protein_total_diff"])
for protein in prot_list:
    total_score = sum(df[df["protein_Id"] == protein]["min_diff"])
    protein_score_df.loc[len(protein_score_df)] = [protein, total_score]
protein_score_df

data_with_protein_score = pd.merge(df, protein_score_df, on="protein_Id", how="left")
data_with_protein_score

In [ ]:
# data_with_protein_score.to_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx", index = False)

# IMPORT HERE THE DATAFRAME WITH DIFFERENCES

In [ ]:
kinase_df = pd.read_excel("Experiment/Kinase_list.xlsx")
kinase_list = list(kinase_df.uniprot_id.unique())
# print(kinase_list)

In [ ]:
df = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")
# df = data_with_protein_score.copy()
df_for_kinases = df.loc[df["protein_Id"].isin(kinase_list)]
df_for_kinases

In [ ]:
# Protein subset
protein = "GAB1"
protein_subset = df.loc[df['protein_name'] == protein] #, ["protein_Id", 'protein_name', 'site', 'difference_EGF_vs_EGFnINS', 'difference_INS_vs_EGFnINS', "difference_geometric_mean"]]

protein_subset_top = protein_subset.sort_values(by=['min_diff'], ascending=False).head(5)

plot_protein_profile(df=df, experiment="hme1_2",
                     proteins = ['AKT2'],
                     saving_path= "", saving_info="",
                     legend="",
                     save_pdf=False, save_png=False)


In [ ]:
#Dataframe subset
subset = df[df["difference_geometric_mean"] > 0.5] #min_diff, protein_total_diff, difference_geometric_mean

subset_sorted = subset.sort_values(by=['difference_geometric_mean'], ascending=False)

subset_sorted_top = subset_sorted.head(25)

protein_list = subset_sorted_top.protein_name.unique()
print(protein_list)
# protein_list = ['']
plot_protein_profile(subset_sorted_top, experiment= "hme1_2" , proteins = protein_list,
                     saving_path = "", saving_info= "",
                     legend = False,
                     save_pdf=False, save_png=False)

# ["AKT2", 'RAF1', 'IGF1R', 'EGFR', 'MAPK3', 'MAP3K1']

In [ ]:
# Checking proteins with the most change between conditions
prot_subset = df_for_kinases[["protein_Id", "protein_total_diff"]]
prot_subset_unique = prot_subset.drop_duplicates()
prot_subset_unique_sorted = prot_subset_unique.sort_values(by=['protein_total_diff'], ascending=False)
top_different_protfile_proteins = list(prot_subset_unique_sorted.protein_Id.head(10))
# top_different_protfile_proteins

# df_filtered = df.loc[df["n_rep"]>1]

plot_protein_profile(df, experiment= "hme1_2" , proteins = top_different_protfile_proteins,
                     saving_path = "", saving_info= "",
                     legend = False,
                     save_pdf=False, save_png=False)

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2
# Compare geometric_mean and min_diff
# geometric mean
gm_subset = df[df["difference_geometric_mean"] > 1]
# gm_subset_sorted = gm_subset.sort_values(by=['difference_geometric_mean'], ascending=False)
# gm_top_sites = list(gm_subset_sorted.site.unique())
gm_subset_sorted_top = gm_subset.sort_values(by=['difference_geometric_mean'], ascending=False).head(100)
gm_top_sites_top = list(gm_subset_sorted_top.site.unique())


# min diff
md_subset = df[df["min_diff"] > 1]
# md_subset_sorted = md_subset.sort_values(by=['min_diff'], ascending=False)
# md_top_sites = list(gm_subset_sorted.site.unique())
md_subset_sorted_top = md_subset.sort_values(by=['min_diff'], ascending=False).head(100)
md_top_sites_top = list(md_subset_sorted_top.site.unique())

# #Do comparison
# #Convert lists to sets
# set1 = set(gm_top_sites)
set1 = set(gm_top_sites_top)
# set2 = set(md_top_sites)
set2 = set(md_top_sites_top)

# Find common elements
common_elements = set1 & set2

# Print results
print(f"Number of common elements: {len(common_elements)}")
print(f"Common elements: {sorted(common_elements)}")

# Create a Venn diagram
plt.figure(figsize=(6, 4))
venn2([set1, set2], set_labels=('Geometric mean top sites', 'Min difference top sites'))
plt.title("Venn Diagram")
plt.show()


# Code to plot simplifyed protein profiles

In [ ]:
# import data
# df = pd.read_excel("Experiment/hek_1/Data/Processed/Full_dataset_HEK293T_diff.xlsx")
# df = pd.read_excel("Experiment/hek_1/Data/Processed/Full_dataset_HEK293T_diff.xlsx")
# df = pd.read_excel("Experiment/hek_1/Data/Processed/Full_dataset_HEK293T_diff.xlsx")
df = pd.read_excel('Experiment/hme1_1/Data/Processed/Full_dataset_1_hTERT_HME1_diff.xlsx')
df = df.fillna(0)
df


In [ ]:
# Filter data
# Filtering only "FC_" columns, excluding those with "statistics"
fc_columns = [col for col in df.columns if col.startswith("FC_") and "stats" not in col.lower() and "pvalue" not in col.lower() and "fdr" not in col.lower()]

# df = df.loc[df["localized_sites"] > 0]
df = df.loc[df["n_rep"] > 1]
# df = df.loc[df["n_reps"] > 1]
#
# fc_columns = list(df.columns)
# fc_columns.remove("protein_ID")
# fc_columns.remove("prot_name")
# fc_columns.remove("site")
# fc_columns.remove("n_reps")
# print(fc_columns)

# Apply filtering condition on selected columns
subset = df[fc_columns]
# subset = subset.head(5).copy()

row_max = subset.max(axis=1)
row_min = subset.min(axis=1)

mask_extremes = (row_max > 0.5) | (row_min < -0.5)

# Step 4: Filter rows into two DataFrames
df_over_05 = df[mask_extremes]
df_below_05 = df[~mask_extremes]

print(len(df), len(df_over_05), len(df_below_05), (len(df_over_05) + len(df_below_05)))

In [ ]:
#plot protein profiles
plot_protein_profile(df=df_over_05, experiment="hme1_1",
                     proteins = ['EGFR', 'BRAF', 'MAPK3'],
                     saving_path= "", saving_info="",
                     legend="",
                     save_pdf=False, save_png=False)

# Q9C086_INO80B
# CERT1

I have to add the difference value to the big dataframe, to order that, do subdataframe and the plot only those site to not have too much over information


In [ ]:
df_diff = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff_renamed.xlsx")
df_diff

In [ ]:
df = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1_diff.xlsx")
df2 = pd.read_excel("Experiment/hme1_2/Data/Processed/Full_dataset_2_hTERT_HME1.xlsx")
# df.loc[df['protein_name', 'site', 'difference_EGF_vs_EGFnINS', 'difference_INS_vs_EGFnINS', "difference_geometric_mean"]]

In [ ]:
df_cut = df.loc[500:1000, :]


In [ ]:
print(df['site'].head(10))


In [ ]:

print(df['site'].tolist())